# 02_run_agent_outcomes_clean

**Purpose**
This notebook runs the outcomes agent on a selected dataset representation (**raw**, **struct**, or **sem**),
and produces a single, analysis-ready table with one row per question × dataset.

**Inputs**
- `../data/input/buddi_questions.csv`
- One of:
  - `../data/processed/raw/` (agent_mode=`"raw"`)
  - `../data/processed/struct/` (agent_mode=`"struct"`)
  - `../data/processed/sem/` (agent_mode=`"sem"`)

**Outputs**
- `../data/processed/buddi_agent_outcomes_<agent_mode>.csv` (agent answers + assumption flags)
- `../outputs/logs/buddi_agent_run_manifest_<agent_mode>.json` (hashes + package versions)

**Unit of analysis**
Question × dataset representation. Each question is evaluated once per dataset mode you run.

**Reproducibility**
- All transformations and prompt assembly are deterministic given fixed inputs.
- We fail loudly on missing files / schema mismatches to avoid silently running on the wrong dataset.

In [1]:
import importlib.metadata as m
print("langchain:", m.version("langchain"))
print("langchain-core:", m.version("langchain-core"))
print("langchain-openai:", m.version("langchain-openai"))
print("langsmith:", m.version("langsmith"))
print("langchain-experimental:", m.version("langchain-experimental"))

langchain: 0.3.27
langchain-core: 0.3.83
langchain-openai: 0.3.35
langsmith: 0.4.37
langchain-experimental: 0.3.4


In [ ]:
# =========================
# Configuration
# =========================
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Optional, Dict, List, Tuple
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain.globals import set_debug

import pandas as pd

# Pick which representation to run
AGENT_MODE = "sem"  # "raw", "struct", "sem"

# Base folders 
PROJECT_ROOT = Path("..")
CLEAN_OUT_ROOT = PROJECT_ROOT / "data" / "processed" 
QUESTIONS_PATH = PROJECT_ROOT / "data" / "input" / "buddi_question_set.csv"


SOURCE_DIRS = {
    "raw": CLEAN_OUT_ROOT / "raw",
    "struct": CLEAN_OUT_ROOT / "struct",
    "sem": CLEAN_OUT_ROOT / "sem",
}

DATASET_FOLDER = SOURCE_DIRS[AGENT_MODE]

DATASET_FILES_BY_MODE = {
    "raw": [
        "raw_BUDDI_patients.csv",  
        "raw_BUDDI_export_neuropsychiatric_history.csv",
        "raw_BUDDI_PROM_angst_export.csv",
        "raw_BUDDI_PROM_cognitief_functioneren_export.csv",
        "raw_BUDDI_PROM_depressieve_klachten_export.csv",
        "raw_BUDDI_PROM_vermoeidheid_export.csv",
        "raw_BUDDI_RBS.csv",
        "raw_BUDDI_SRS.csv",
        "raw_BUDDI_Vineland.csv",
    ],
    "struct": [
        "struct_BUDDI_patients.csv",
        "struct_BUDDI_export_perinatal_history.csv",
        "struct_BUDDI_export_screening_lab.csv",
        "struct_BUDDI_export_neuropsychiatric_history.csv",
        "struct_BUDDI_PROM_angst_export.csv",
        "struct_BUDDI_PROM_cognitief_functioneren_export.csv",
        "struct_BUDDI_PROM_depressieve_klachten_export.csv",
        "struct_BUDDI_PROM_vermoeidheid_export.csv",
        "struct_BUDDI_RBS.csv",
        "struct_BUDDI_SRS.csv",
        "struct_BUDDI_Vineland.csv",
    ],
    "sem": [
        "sem_BUDDI_patients.csv",
        "sem_BUDDI_export_perinatal_history.csv",
        "sem_BUDDI_export_screening_lab.csv",
        "sem_BUDDI_export_neuropsychiatric_history.csv",
        "sem_BUDDI_PROM_angst_export.csv",
        "sem_BUDDI_PROM_cognitief_functioneren_export.csv",
        "sem_BUDDI_PROM_depressieve_klachten_export.csv",
        "sem_BUDDI_PROM_vermoeidheid_export.csv",
        "sem_BUDDI_RBS.csv",
        "sem_BUDDI_SRS.csv",
        "sem_BUDDI_Vineland.csv",
    ],
}

# This is the *only* prompt line to change for raw vs struct vs sem. Make sure to change this in /prompt/system_prompt.txt when you change modes.
DATASET_TYPE_LINE: Dict[str, str] = {
    "raw": (
        "Foundational: raw unstandardized export"
    ),
    "struct": (
        "Structured: dataframes have been structured into a flattened FHIR format"
    ),
    "sem": (
        "Structured & Semantic: dataframes have been structured into a flattened FHIR format and harmonized in semantics (e.g. with SNOMED codes and standardized labels)"
    ),
}


# Output path
OUT_DIR = Path("../data/agent_outcomes")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / f"agent_outcomes_{AGENT_MODE}.csv"

print("AGENT_MODE:", AGENT_MODE)
print("DATASET_FOLDER:", DATASET_FOLDER.resolve())
print("QUESTIONS_PATH:", QUESTIONS_PATH.resolve())
print("OUT_PATH:", OUT_PATH.resolve())


AGENT_MODE: sem
DATASET_FOLDER: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/processed/sem
QUESTIONS_PATH: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/input/buddi_question_set.csv
OUT_PATH: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/data/agent_outcomes/agent_outcomes_sem.csv


In [3]:
# =========================
# Helpers 
# =========================
def load_dataset_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=None, engine="python", dtype=str)

def extract_assumption_flags(answer_text: str) -> tuple[str, str]:
    """Best-effort parsing to yes/no flags from the model's answer.

    This does NOT change the agent output; it just derives the two yes/no fields
    you asked for. Adjust the regexes later if you standardize the agent output.
    """
    if answer_text is None:
        return ("no", "no")
    t = str(answer_text)

    def _flag(section_name: str) -> str:
        # Look for a section header and see if it contains a 'none'/'no assumption' marker.
        m = re.search(rf"{section_name}\s*[:\-]\s*(.*)", t, re.IGNORECASE)
        if not m:
            return "no"
        rest = m.group(1).strip()
        if rest == "" or re.match(r"^(none|no|n/a)\b", rest, re.IGNORECASE):
            return "no"
        return "yes"

    structural = _flag("Structural Assumption")
    semantic = _flag("Semantic Assumption")
    return structural, semantic


In [10]:
# =========================
# Agent  
# =========================
from utils import get_chat

from langchain_experimental.agents import create_pandas_dataframe_agent

PROMPT_PATH = PROJECT_ROOT / "prompt" / "system_prompt.txt"
PREFIX_TEMPLATE = PROMPT_PATH.read_text()

os.environ["LANGCHAIN_PROJECT"] = "BUDDI_Project_" + AGENT_MODE


def create_agent(*, dataset_folder: Path, agent_mode: str):
    """Create the pandas DataFrame agent.
    Only the 'Dataset type: ...' line changes via DATASET_TYPE_LINE[agent_mode].
    """
    if agent_mode not in DATASET_TYPE_LINE:
        raise ValueError(f"Unknown agent_mode={agent_mode!r}. Expected one of {list(DATASET_TYPE_LINE)}")
        files = DATASET_FILES_BY_MODE[agent_mode]

    files = DATASET_FILES_BY_MODE[agent_mode]

    missing = [fn for fn in files if not (dataset_folder / fn).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing expected files in dataset folder:\n"
            + str(dataset_folder.resolve())
            + "\n\nMissing:\n- "
        + "\n- ".join(missing)
    )

    dfs = [load_dataset_csv(dataset_folder / fn) for fn in files]


    llm = get_chat()

    prefix = PREFIX_TEMPLATE.format(dataset_type=DATASET_TYPE_LINE[agent_mode])

    agent_executor = create_pandas_dataframe_agent(
        llm,
        dfs,
        verbose=True,
        agent_type="tool-calling",
        prefix=prefix,
        max_iterations=15,
        allow_dangerous_code=True,
        #handle_parsing_errors=True,
    )

    return agent_executor

set_debug(False)

agent = create_agent(dataset_folder=DATASET_FOLDER, agent_mode=AGENT_MODE)
print("Agent ready.")


Agent ready.


In [5]:
# =========================
# Run agent and collect outcomes
# =========================
df_questions = pd.read_csv(QUESTIONS_PATH)

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

rows = []
for _, qrow in tqdm(df_questions.iterrows(), total=len(df_questions)):
    qid = qrow["question_id"]
    theme = qrow["question_theme"]
    complexity = qrow["complexity_label"]
    question = qrow["question"]

    try:
        resp = agent.invoke({"input": question})
        answer = resp.get("output") if isinstance(resp, dict) else str(resp)
    except Exception as e:
        answer = f"[ERROR] {type(e).__name__}: {e}"

    structural_flag, semantic_flag = extract_assumption_flags(answer)

    rows.append({
        "question_id": qid,
        "dataset": AGENT_MODE,
        "question_theme": theme,
        "complexity_label": complexity,
        "answer": answer,
        "structural_assumption_made": structural_flag,
        "semantic_assumption_made": semantic_flag,
    })

df_outcomes = pd.DataFrame(rows)
df_outcomes.head()


  0%|          | 0/25 [00:00<?, ?it/s]



> Entering new AgentExecutor chain...
Answer:  
There are 9 datasources (pandas DataFrames) administered into memory, as evidenced by 9 different dataframe head printouts provided.

Complexity Level: 1, General Orientation — The question is about the number of DataFrames present, which can be answered by general dataset knowledge without dataframe inspection.
DataFrame(s) Used: All (general enumeration)
Columns Used: None
Structural Assumption: No — All datasources are explicitly shown with a .head() representative, so count is not assumed.
Semantic Assumption: No — No semantic interpretation was required beyond general dataset structure.

> Finished chain.


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': '# Inspect the relevant DataFrame for Vineland scores\n# The correct DataFrame contains Vineland adaptive behavior scores\n# It has columns: component.valueQuantity.code.text (score type), component.valueQuantity.code.coding.display (subdomain), s

<string>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


Answer:  
55.5% of patients (66 out of 119) ever had a potassium measurement < 3.5 mmol/l at any point in time.

Complexity Level: 4, Multi-step analytical — Requires matching potassium measurements, normalizing patient IDs, and aggregation across dataframes.
DataFrame(s) Used: [Patient demographic dataframe, Observation dataframe with lab results]
Columns Used: ['id', 'Observation.code.coding.text', 'Observation.code.coding.display', 'Observation.valueQuantity.value', 'Observation.subject.reference']
Structural Assumption: Yes — Assumed all potassium lab results are captured in the Observation dataframe via code.text prefixes ("lab_k") or display ("Potassium level").
Semantic Assumption: Yes — Assumed measurements labeled via code.text prefix "lab_k*" and/or display “Potassium level” are all serum/plasma potassium values in mmol/l, and threshold of 3.5 is clinically relevant and consistent.

> Finished chain.


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{

<string>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Invoking: `python_repl_ast` with `{'query': '# Now, get PROM fatigue baseline T-scores for these IDs\n# Review columns and data first\ndf7.columns, df7.head()'}`


(Index(['Participant Id', 'Participant Status', 'Repeating Data Creation Date',
       'Repeating data Name Custom', 'Repeating data Parent',
       'PROM_depressieve_klachten_date', 'PROM_depressieve_klachten_Invuller',
       'PROM_depressieve_klachten_Tscore', 'PROM_depressieve_klachten_SE'],
      dtype='object'),   Participant Id Participant Status Repeating Data Creation Date  \
0          BU008            Not Set          2023-05-12 13:46:14   
1          BU011            Not Set          2023-05-12 13:46:25   
2          BU030            Not Set          2023-05-12 13:46:51   
3          BU065            Not Set          2023-05-12 13:47:11   
4          BU026            Not Set          2024-04-18 16:58:27   

        Repeating data Name Custom Repeating data Parent  \
0             depressie_28-02-2021            

,question_id,dataset,question_theme,complexity_label,answer,structural_assumption_made,semantic_assumption_made
0,1,sem,Development,1,Answer: \nThere are 9 datasources (pandas Dat...,no,no
1,2,sem,Development,2,Answer: \nThe Vineland V-score for coping ski...,no,no
2,3,sem,Development,3,Answer: \nThe standard deviation of the patie...,no,yes
3,4,sem,Development,4,Answer: \n- Number of unique children with a ...,no,no
4,5,sem,Development,5,Answer: \nCannot answer the question as posed...,yes,no


In [6]:
OUT_DIR = PROJECT_ROOT / "outputs" / "agent_outcomes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"buddi_agent_outcomes_{AGENT_MODE}.csv"

df_outcomes.to_csv(out_path, index=False)

display(df_outcomes.head())
print(f"Saved: {out_path}")

,question_id,dataset,question_theme,complexity_label,answer,structural_assumption_made,semantic_assumption_made
0,1,sem,Development,1,Answer: \nThere are 9 datasources (pandas Dat...,no,no
1,2,sem,Development,2,Answer: \nThe Vineland V-score for coping ski...,no,no
2,3,sem,Development,3,Answer: \nThe standard deviation of the patie...,no,yes
3,4,sem,Development,4,Answer: \n- Number of unique children with a ...,no,no
4,5,sem,Development,5,Answer: \nCannot answer the question as posed...,yes,no


Saved: /Users/feline.mollerus/Documents/jbhi_llm_paper/buddi_project/outputs/agent_outcomes/buddi_agent_outcomes_sem.csv
